# RL Masterclass 05: Proximal Policy Optimization (PPO)

**Track:** Reinforcement Learning (0 to Masterclass)  
**Prerequisites:** RL 04 (Policy Gradient / Actor-Critic)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"REINFORCE updates the policy once per episode. But what if we could squeeze MORE learning from the same collected data by updating MULTIPLE times? The problem: too many updates and the policy changes so much that the old data becomes invalid. PPO solves this with a clever trick. What is it?"*

---

## Why PPO Is the Most Important RL Algorithm

PPO (Schulman et al., 2017) is the DEFAULT algorithm for:

| Application | Why PPO |
|------------|--------|
| **RLHF (ChatGPT, Claude)** | Aligns LLMs to human preferences (NB12) |
| **OpenAI Five (Dota 2)** | First AI to beat world champions |
| **Robotics (sim-to-real)** | Stable enough for physical robots |
| **Game AI** | Default in Unity ML-Agents |

PPO is popular because it's **simple to implement**, **stable to train**, and **works across many domains**. If you only learn one RL algorithm deeply, make it PPO.

## Table of Contents
1. The Trust Region Problem
2. PPO-Clip: The Core Mechanism
3. GAE: Generalized Advantage Estimation
4. Complete PPO Implementation

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** PPO's genius is making policy gradient methods PRACTICAL. Before PPO, policy gradient methods were fragile — one bad update could destroy the policy. PPO adds a safety mechanism (the clip) that prevents catastrophic policy changes while still allowing efficient learning from batched data.

**Mental Model:** Imagine adjusting a satellite dish to get better TV signal. REINFORCE moves the dish once per signal test. PPO takes multiple readings, then makes several small adjustments — but with a safety lock that prevents turning the dish more than 20% from its current position. This way you improve faster without losing the signal entirely.

**Common Junior Pitfall:** Setting the clip range too large (ε > 0.3). This defeats PPO's purpose — the policy can change too much per update, leading to instability. The standard ε=0.2 was tuned extensively and rarely needs changing.

---

In [ ]:
!pip install -q torch numpy matplotlib

## 1 · The Trust Region Problem

### Why We Need PPO

**Problem with REINFORCE/Actor-Critic:** One gradient step. Collect data → compute gradient → update → throw away data. Wasteful!

**Naive fix:** Update multiple times on the same data.

**Why this breaks:** After the first update, the policy has CHANGED. Data was collected under the OLD policy. Using old data with the new policy creates a mismatch. Too many updates → policy diverges.

### The Solution: Trust Regions

Allow multiple updates, but CONSTRAIN how much the policy can change:

| Method | How It Constrains | Complexity |
|--------|------------------|------------|
| **TRPO** (2015) | KL-divergence constraint | Complex (requires conjugate gradient) |
| **PPO-Clip** (2017) | Clips probability ratio | Simple (just a min/clip) |
| **PPO-Penalty** | KL penalty term | Medium |

PPO-Clip won because it's the simplest and works just as well.

---
## 2 · PPO-Clip: The Core Mechanism

### The Probability Ratio

$$r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{\text{old}}}(a_t | s_t)}$$

- $r_t = 1$: new policy same as old → no change
- $r_t > 1$: new policy makes this action MORE likely
- $r_t < 1$: new policy makes this action LESS likely

### The Clipped Objective

$$L^{\text{CLIP}}(\theta) = \mathbb{E} \left[ \min\left( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$

In plain English with ε=0.2:
- If the action was GOOD ($\hat{A} > 0$): allow increasing its probability, but cap the increase at 1.2× the old probability
- If the action was BAD ($\hat{A} < 0$): allow decreasing its probability, but cap the decrease at 0.8× the old probability
- Result: the policy changes gradually, never making drastic jumps

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Visualize the PPO clip mechanism
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epsilon = 0.2
ratios = np.linspace(0.5, 1.8, 200)

for ax, advantage, title in [(axes[0], 1.0, 'Good Action (A > 0)'),
                              (axes[1], -1.0, 'Bad Action (A < 0)')]:
    # Unclipped objective
    unclipped = ratios * advantage
    # Clipped objective
    clipped_ratios = np.clip(ratios, 1 - epsilon, 1 + epsilon)
    clipped = clipped_ratios * advantage
    # PPO objective: min of both
    ppo_obj = np.minimum(unclipped, clipped) if advantage > 0 else np.maximum(unclipped, clipped)
    # For A > 0, min clips the upside. For A < 0, we want max of negative values (less negative)
    ppo_obj = np.where(advantage > 0, np.minimum(unclipped, clipped), np.maximum(unclipped, clipped))
    
    ax.plot(ratios, unclipped, label='Unclipped', lw=2, linestyle='--', alpha=0.5)
    ax.plot(ratios, ppo_obj, label='PPO-Clip', lw=3, color='steelblue')
    ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(x=1-epsilon, color='red', linestyle='--', alpha=0.5, label=f'1-ε={1-epsilon}')
    ax.axvline(x=1+epsilon, color='red', linestyle='--', alpha=0.5, label=f'1+ε={1+epsilon}')
    ax.set_xlabel('Probability Ratio r(θ)')
    ax.set_ylabel('Objective')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('PPO Clipping: Prevents Policy from Changing Too Much', fontweight='bold')
plt.tight_layout()
plt.show()

print('Left plot (good action, A>0):')
print('  Without clip: ratio can go to 2x, 5x — dangerously large update')
print('  With clip: ratio capped at 1.2 — controlled improvement')
print()
print('Right plot (bad action, A<0):')
print('  Without clip: ratio can go to 0 — might remove action entirely')
print('  With clip: ratio capped at 0.8 — gradual reduction')

---
## 3 · GAE: Generalized Advantage Estimation

The advantage $A_t$ in PPO uses **GAE** (Generalized Advantage Estimation) for the best bias-variance tradeoff:

$$\hat{A}_t^{\text{GAE}} = \sum_{l=0}^{T-t} (\gamma \lambda)^l \delta_{t+l}$$

where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD error.

| λ value | Behavior | Bias | Variance |
|---------|----------|------|----------|
| λ=0 | One-step TD (pure critic) | High | Low |
| λ=1 | Monte Carlo (full returns) | Low | High |
| **λ=0.95** | **Best tradeoff (default)** | **Medium** | **Medium** |

Think of λ as controlling how far into the future the advantage calculation looks.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """Generalized Advantage Estimation (GAE).
    
    This is the advantage estimator used in PPO and RLHF.
    
    Args:
        rewards: list of rewards [r_0, r_1, ..., r_T]
        values:  list of V(s) estimates [V(s_0), ..., V(s_T), V(s_{T+1})]
        dones:   list of done flags
        gamma:   discount factor
        lam:     GAE lambda (bias-variance tradeoff)
    
    Returns:
        advantages: GAE advantage estimates
        returns:    targets for value function training
    """
    T = len(rewards)
    advantages = np.zeros(T)
    gae = 0
    
    for t in reversed(range(T)):
        next_value = values[t + 1] if t + 1 < len(values) else 0
        delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages[t] = gae
    
    returns = advantages + np.array(values[:T])  # V(s) + A(s,a) = Q(s,a) ≈ return
    return advantages, returns

# Demo
rewards = [1, 1, 1, 1, 10]  # Big reward at the end
values =  [2, 3, 4, 5, 8, 0]  # V(s) estimates (last is terminal)
dones =   [0, 0, 0, 0, 1]

advantages, returns = compute_gae(rewards, values, dones)
print('GAE Computation:')
print(f'  Rewards:    {rewards}')
print(f'  Values:     {values[:-1]}')
print(f'  Advantages: {advantages.round(3).tolist()}')
print(f'  Returns:    {returns.round(3).tolist()}')
print(f'\n  Last step has highest advantage — the big reward was unexpected (10 vs V=8).')
print(f'  GAE propagates this "surprisingly good" signal backward through time.')

---
## 4 · Complete PPO Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class PPOActorCritic(nn.Module):
    """Actor-Critic network for PPO."""
    def __init__(self, state_dim, action_dim, hidden=64):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, action_dim), nn.Softmax(dim=-1),
        )
        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
    
    def forward(self, state):
        return self.actor(state), self.critic(state)
    
    def act(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0)
        probs, value = self.forward(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action).item(), value.item()

class SimpleBalanceEnv:
    def __init__(self):
        self.reset()
    def reset(self):
        self.state = np.random.uniform(-0.05, 0.05, size=4)
        self.steps = 0
        return self.state.copy()
    def step(self, action):
        force = 1.0 if action == 1 else -1.0
        self.state[3] += force * 0.02 + np.random.normal(0, 0.01)
        self.state[2] += self.state[3] * 0.02
        self.state[1] += force * 0.01
        self.state[0] += self.state[1] * 0.02
        self.steps += 1
        done = abs(self.state[2]) > 0.5 or abs(self.state[0]) > 2.0 or self.steps >= 200
        reward = 1.0 if not done else 0.0
        return self.state.copy(), reward, done

def ppo_train(env, total_episodes=500, gamma=0.99, lam=0.95,
              clip_epsilon=0.2, epochs_per_update=4, lr=3e-4):
    """PPO Training Loop.
    
    The key difference from REINFORCE:
    1. Collect a batch of episodes
    2. Compute advantages with GAE
    3. Update policy MULTIPLE TIMES on the same data (with clipping)
    """
    model = PPOActorCritic(state_dim=4, action_dim=2)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    all_rewards = []
    
    for episode in range(total_episodes):
        # --- Step 1: Collect trajectory ---
        states, actions, rewards, log_probs_old, values, dones = [], [], [], [], [], []
        state = env.reset()
        
        while True:
            action, log_prob, value = model.act(state)
            next_state, reward, done = env.step(action)
            
            states.append(state)
            actions.append(action)
            rewards.append(reward)
            log_probs_old.append(log_prob)
            values.append(value)
            dones.append(done)
            
            state = next_state
            if done:
                break
        
        all_rewards.append(sum(rewards))
        
        # --- Step 2: Compute GAE advantages ---
        values.append(0)  # Terminal value
        advantages, returns = compute_gae(rewards, values, dones, gamma, lam)
        
        # Convert to tensors
        states_t = torch.FloatTensor(np.array(states))
        actions_t = torch.LongTensor(actions)
        old_log_probs_t = torch.FloatTensor(log_probs_old)
        advantages_t = torch.FloatTensor(advantages)
        returns_t = torch.FloatTensor(returns)
        
        # Normalize advantages
        if len(advantages_t) > 1:
            advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)
        
        # --- Step 3: PPO update (MULTIPLE epochs on same data!) ---
        for _ in range(epochs_per_update):
            probs, values_pred = model(states_t)
            dist = torch.distributions.Categorical(probs)
            new_log_probs = dist.log_prob(actions_t)
            entropy = dist.entropy().mean()
            
            # Probability ratio: π_new / π_old
            ratio = torch.exp(new_log_probs - old_log_probs_t)
            
            # PPO Clipped objective
            surr1 = ratio * advantages_t
            surr2 = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * advantages_t
            actor_loss = -torch.min(surr1, surr2).mean()
            
            # Critic loss: MSE between predicted and actual returns
            critic_loss = nn.functional.mse_loss(values_pred.squeeze(), returns_t)
            
            # Total loss = actor + critic - entropy bonus
            loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy
            
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
    
    return model, all_rewards

# Train PPO
env = SimpleBalanceEnv()
print('Training PPO...')
ppo_model, ppo_rewards = ppo_train(env, total_episodes=400)

# Plot results
window = 30
smoothed = [np.mean(ppo_rewards[max(0,i-window):i+1]) for i in range(len(ppo_rewards))]

plt.figure(figsize=(10, 5))
plt.plot(smoothed, lw=2, color='steelblue', label='PPO')
plt.axhline(y=200, color='green', linestyle='--', alpha=0.7, label='Maximum (200)')
plt.xlabel('Episode')
plt.ylabel('Total Reward (smoothed)')
plt.title('PPO Training: Balance Task')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Early: {np.mean(ppo_rewards[:30]):.0f} steps | Late: {np.mean(ppo_rewards[-30:]):.0f} steps')
print(f'PPO learned to balance! This is the SAME algorithm that fine-tunes ChatGPT.')

---
## PPO Hyperparameters: What to Tune

| Hyperparameter | Default | Effect of Too High | Effect of Too Low |
|---|:-:|---|---|
| **clip ε** | 0.2 | Policy changes too fast (unstable) | Policy barely changes (slow learning) |
| **GAE λ** | 0.95 | Higher variance (noisy) | Higher bias (inaccurate advantage) |
| **Epochs per update** | 3-10 | Overfits to old data | Wastes data |
| **Learning rate** | 3e-4 | Diverges | Too slow |
| **Entropy coefficient** | 0.01 | Too random (won't converge) | No exploration (premature convergence) |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *How does the clip prevent policy collapse while allowing multiple updates?*

**A:** The clip creates a "trust region" — a zone around the old policy where updates are allowed. After the first update, the ratio might be 1.1 (policy changed 10%). After the second update, it might try to go to 1.3, but the clip caps it at 1.2. Subsequent updates produce zero gradient for the clipped samples, effectively stopping further changes for those data points. The policy can only change within the trust region (1 ± ε), no matter how many update epochs you run.

---

## ✅ Knowledge Check

### Q1: PPO vs REINFORCE
PPO updates 4-10 times per batch of data. REINFORCE updates once and discards the data. Why is PPO more sample-efficient?

<details><summary>👉 Answer</summary>

PPO extracts more learning from each collected trajectory by performing multiple gradient updates. The clip mechanism ensures these extra updates don't destabilize training. REINFORCE wastes data by using each experience only once. In practice, PPO needs 3-10x fewer environment interactions to reach the same performance.
</details>

### Q2: Entropy bonus
PPO subtracts 0.01 * entropy from the loss. Why does encouraging randomness help training?

<details><summary>👉 Answer</summary>

Without the entropy bonus, the policy can become deterministic too quickly — always choosing the same action. This kills exploration. The entropy bonus rewards policies that maintain some uncertainty, keeping exploration alive. As training progresses, the advantage signal naturally overwhelms the small entropy bonus, and the policy sharpens. It's a self-annealing exploration strategy.
</details>

### Q3: Connection to RLHF
In RLHF, PPO has an additional KL penalty term: loss += β * KL(π_new || π_ref). Why?

<details><summary>👉 Answer</summary>

Without the KL penalty, the RLHF-trained LLM would drift far from the original pre-trained model — it might find "hacks" that get high reward model scores but produce gibberish text. The KL penalty keeps the fine-tuned model close to the reference model, preserving language quality while improving alignment. This is the same concept as the PPO clip but applied at the distribution level.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Clip Ablation
Train PPO with ε = 0.05, 0.1, 0.2, 0.3, 0.5. Compare learning curves. Which is most stable? Which is fastest?

### Exercise 2: Continuous Actions
Modify PPO for continuous action spaces. The actor outputs (mean, log_std) of a Gaussian, and you sample actions from N(mean, std). Use this for a pendulum swing-up task.

### Exercise 3: Multi-Environment PPO
Run N copies of the environment in parallel, collecting data from all simultaneously. This is how production PPO works (vectorized environments = linear speedup).

---

**Next →** RL 06: RLHF — From PPO to Language Model Alignment